# Cleaning details.csv

In [19]:
import pandas as pd
import os
from unidecode import unidecode
import ast

In [20]:
# path
RAW_PATH = os.path.join("../..", "dataset", "raw", "details.csv")
CLEAN_PATH = os.path.join("../..", "dataset", "cleaned", "details_cleaned.csv")

## Pulizia strutturale ID

In [21]:
df = pd.read_csv(RAW_PATH)
original_rows = len(df)

df = df.dropna(subset=['mal_id'])
df['mal_id'] = pd.to_numeric(df['mal_id'], errors='coerce').astype(int)

## Gestione valori mancanti

In [22]:
df['title'] = df['title'].fillna("Unknown")
df['type'] = df['type'].fillna("Unknown")
df['status'] = df['status'].fillna("Unknown")
df['source'] = df['source'].fillna("Unknown")

## Normalizzazione testo

In [23]:
def normalize_text(x):
    if isinstance(x, str):
        return unidecode(x).strip().title()
    return x

df['title'] = df['title'].apply(normalize_text)
df['type'] = df['type'].apply(normalize_text)
df['status'] = df['status'].apply(normalize_text)
df['source'] = df['source'].apply(normalize_text)


## Conversione tipi

In [24]:
# string
df['type'] = df['type'].astype("string")
df['status'] = df['status'].astype("string")
# numeric
df['score'] = pd.to_numeric(df['score'], errors='coerce')
df['scored_by'] = pd.to_numeric(df['scored_by'], errors='coerce')
df['rank'] = pd.to_numeric(df['rank'], errors='coerce')
# datetime
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')

## Pulizia variabili multivalore - genres

In [25]:
# valori mancati -> lista vuota
df['genres'] = df['genres'].fillna("[]")

def parse_genres(x):
    # se è gia una lista, la restituiamo
    if isinstance(x, list):
        return x
    # convertiamo la stringa in lista
    try:
        return ast.literal_eval(x)
    except:
        return []
# parsing stringa -> lista
df['genres'] = df['genres'].apply(parse_genres)
# se non ci sono generi, rendiamo esplicita l'incertezza
df['genres'] = df['genres'].apply(
    lambda g: ['Unknown'] if len(g) == 0 else g
)
# normalizzazione dei nomi di
df['genres'] = df['genres'].apply(
    lambda g: [genre.strip().title() for genre in g]
)

## Pulizia variabili multivalore -studios

In [26]:
def parse_studios(x):
    if pd.isna(x):
        return ['Unknown']
    if isinstance(x, str):
        x = x.strip()
        try:
            lst = ast.literal_eval(x)
            return [s.strip().title() for s in lst if s.strip()]
        except:
            return [s.strip().title() for s in x.split(',') if s.strip()]
    return ['Unknown']

df['studios_list'] = df['studios'].apply(parse_studios)


## Duplicati

In [27]:
#df = df.drop_duplicates()
df = df.drop_duplicates(subset=['mal_id'])

print("Righe iniziali:", original_rows)
print("Righe finali:", len(df))
print("Righe rimosse:", original_rows - len(df))

df.to_csv(CLEAN_PATH, index=False, encoding="utf-8")


Righe iniziali: 28955
Righe finali: 28955
Righe rimosse: 0
